<a href="https://colab.research.google.com/github/ogrisel/notebooks/blob/master/ridgeclassifier_and_calibration_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPU computing with scikit-learn 1.8

This notebook demonstrates the impact of enabling array API dispatch with scikit-learn 1.8 so as to accelerate computation with GPUs.

In [ ]:
%pip install -q scikit-learn==1.8.0

Let's enable the experimental array API dispatch in both SciPy and scikit-learn:

In [ ]:
import os
os.environ["SCIPY_ARRAY_API"] = "1"

from sklearn import set_config
set_config(array_api_dispatch=True)

Let's define an unbalanced, binary classification problem with a mix of categorical and numerical features and marginally non-linear feature effects.

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import expit

n_rows = 500_000

rng = np.random.default_rng(42)

features = pd.DataFrame({
    'product_category': rng.choice(['A', 'B', 'C'], size=n_rows),
    'location': rng.choice([f'region_{i}' for i in range(50)], size=n_rows),
    'prev_activity': rng.lognormal(mean=0, sigma=0.5, size=n_rows),
}).astype({
    'product_category': 'category',
    'location': 'category',
    'prev_activity': 'float32',
})

def ground_truth_event_probability(features):
    event_logit = -2
    event_logit -= 2 * np.log(features['prev_activity']).clip(1, None)
    event_logit -= 0.1 * (features['product_category'] == 'A').astype(int)
    event_logit += 1 * (features['product_category'] == 'B').astype(int)
    event_logit += 5 * (features['location'] == 'region_1').astype(int)
    event_logit -= 2 * (features['location'] == 'region_9').astype(int)
    event_logit += 10 * (features['location'] == 'region_10').astype(int)
    event_logit -= 3 * (features['location'] == 'region_19').astype(int)
    return expit(event_logit)


target = rng.binomial(n=1, p=ground_truth_event_probability(features))
target.mean()

np.float64(0.060596)

We evaluate our probabilistic classifiers using the D2 Brier score metric to assess both calibration and resolution at the same time:

In [ ]:
from sklearn.model_selection import cross_validate


def evaluate(model, model_name, X, y):
    cv_results = cross_validate(
        model, X, y, scoring="d2_brier_score", error_score="raise"
    )
    test_scores = cv_results["test_score"]
    fit_time = cv_results["fit_time"]
    print(
        f"{model_name}: total fit time: {fit_time.sum():.1f} s, "
        f"D2 Brier score: {test_scores.mean():.3f} +/- {test_scores.std():.3f}"
    )

Let's now define a pipeline that performs rich enough feature engineering to ensure that fitting a linear classifier on top should result in a well specified model.

We turn a `RidgeClassifierCV` model into a probabilistic classifier by wrapping it with `CalibratedClassifierCV`.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import QuantileTransformer, TargetEncoder, SplineTransformer
from sklearn.linear_model import RidgeClassifierCV
from sklearn.calibration import CalibratedClassifierCV


alphas = np.logspace(-6, 6, 31)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            make_pipeline(
                QuantileTransformer(),
                SplineTransformer(n_knots=10),
            ),
            make_column_selector(dtype_include="number"),
        ),
        (
            "categorical",
            TargetEncoder(cv=5),
            make_column_selector(dtype_include="category"),
        ),
    ],
)


ridge_pipeline_cpu = make_pipeline(
    preprocessor,
    CalibratedClassifierCV(
        RidgeClassifierCV(alphas=alphas), method="temperature", cv=5
    ),
)

evaluate(ridge_pipeline_cpu, "Ridge pipeline (CPU)", features, target)

Ridge pipeline (CPU): total fit time: 72.2 s, D2 Brier score: 0.504 +/- 0.006


Note that the feature engineering works with a mix of categorical and numerical features. Once the features are preprocessed by the column transformer, they are all numerical values. We can leverage this fact by moving the results to a GPU device with the help of PyTorch and benefit from the array API support in scikit-learn to accelerate the computation of the second half:

In [ ]:
import torch
from sklearn.preprocessing import FunctionTransformer


ridge_pipeline_gpu = make_pipeline(
    # Since the input data has a mix of categorical and numerical values,
    # the feature engineering happens on the CPU using Pandas and NumPy as
    # usual.
    preprocessor,
    # Move the resulting numerical features to the GPU with torch and perform
    # all the remaining numerical computations there.
    FunctionTransformer(
        lambda x: torch.tensor(x.astype(np.float32), device="cuda")
    ),
    CalibratedClassifierCV(
        RidgeClassifierCV(alphas=alphas), method="temperature", cv=5
    ),
)

evaluate(ridge_pipeline_gpu, "Ridge pipeline (GPU)", features, target)

Ridge pipeline (GPU): total fit time: 8.0 s, D2 Brier score: 0.504 +/- 0.006


Enabling the colab GPU vs using a single colab CPU for the last step of the pipeline decreases the fit time by a factor of ~8 for this particular pipeline.

The resulting score is the same, despite the use of lower precision `float32` values in the GPU pipeline.

Disclaimers:

- Using `CalibratedClassifierCV` with `cv=5` here is slightly overkill: since the dataset is large, a single shuffle split with a calibration set of 1000 points would yield the same D2 Brier score (but the effect of using the GPU would be not as impressive.)

- The ridge classifier could be turned into a valid probabilistic classifier by shifting and clipping its prediction to the `[0, 1]` range. We would not need the `CalibratedClassifierCV` wrapper at all while achieving an even better D2 Brier than our temperature calibrator. This is left for future work.

- The dataset is large, so we could set the regularization parameter to a small value instead of tuning it. This would also enable us to use faster solvers, but those do not yet support the array API.

- Finally, a logistic regression based pipeline with its default `tol` value and histogram gradient boosting on the raw features will learn competitive models with a favorable D2 Brier / fit time tradeoff as well.